# Observer.ai: score one trace and send kept traces to Phoenix

Takes a single agent trace (a plain Python dict), asks the fine-tuned Kev model whether to keep it, and when the decision is **keep**, prints it and exports the trace to Phoenix.

**Flow**
1. Load the model manifest (questions + temperature) from the fine-tuning run
2. Build the trace slice, the same `state` the model was trained on
3. Ask Kev, calibrate, decide keep or drop (plus the random floor)
4. Kept traces go to Phoenix as OpenTelemetry spans, with the decision as span attributes

**Before you start**
- Fine-tuned Kev serving on port 8010: `uv run --extra serve python -m kev.serve --run <RUN_DIR> --port 8010`
- Phoenix running: `uvx --from arize-phoenix phoenix serve` (UI at http://localhost:6006)
- In this notebook's env: `uv add requests opentelemetry-sdk opentelemetry-exporter-otlp-proto-http`

In [1]:
import json, random, time, uuid
from dataclasses import dataclass, asdict
from pathlib import Path

import requests
from opentelemetry import trace as otel_trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

# ---- config ----
KEV_URL          = "http://127.0.0.1:8010/v1/systemone"          # fine-tuned run
MANIFEST_PATH    = Path("runs/observer-kev-0.8b-ft/manifest.json")
PHOENIX_ENDPOINT = "http://localhost:6006/v1/traces"
PHOENIX_PROJECT  = "observer-ai"

DECISION_QUESTION = "answer_wrong"   # the noul question whose probability becomes p(keep)
KEEP_THRESHOLD    = 0.5              # keep when p(keep) >= this (Kev's top choice)
ALERT_THRESHOLD   = 0.8              # alert.severity_high in observer.yaml
RANDOM_FLOOR      = 0.02             # share of traces kept at random, whatever Kev says
TIMEOUT_S         = 10

rng = random.Random()

## 1. Model manifest

Uses the exact questions the model was fine-tuned on, plus the fitted temperature. Falls back to defaults if the manifest is missing.

In [2]:
DEFAULT_QUESTIONS = {
    "answer_wrong": {
        "type": "noul",
        "instructions": "Is the final answer wrong or only partially correct for the user's question?",
    },
}

if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text())
    QUESTIONS = manifest["questions"]
    TEMPERATURE = manifest.get("temperatures", {}).get(DECISION_QUESTION, 1.0)
    MODEL_NAME = manifest.get("model", "kev")
    print(f"manifest: {MODEL_NAME}, temperature {TEMPERATURE}, questions {list(QUESTIONS)}")
else:
    QUESTIONS, TEMPERATURE, MODEL_NAME = DEFAULT_QUESTIONS, 1.0, "kev"
    print("no manifest found, using default question and T = 1.0")

assert DECISION_QUESTION in QUESTIONS and QUESTIONS[DECISION_QUESTION]["type"] == "noul"

manifest: observer-kev-0.8b-ft, temperature 0.8581, questions ['answer_wrong', 'answers_question']


## 2. Trace slice

**Must stay identical** to `build_state` in the fine-tuning notebook. If they drift, the model sees inputs it was never trained on.

In [3]:
MAX_EVIDENCE = 3
MAX_EVIDENCE_CHARS = 1500

def build_state(t):
    tools = "\n".join(
        f"{i}. {c['name']}({c.get('args', '')!r}) -> {c.get('status', 'ok')}"
        for i, c in enumerate(t.get("tool_calls", []), 1)
    ) or "(none)"
    evidence = "\n".join(
        f"[{i}] {e[:MAX_EVIDENCE_CHARS]}" for i, e in enumerate(t.get("evidence", [])[:MAX_EVIDENCE], 1)
    ) or "(none)"
    return (
        f"USER QUESTION:\n{t['question']}\n\n"
        f"TOOL CALLS:\n{tools}\n\n"
        f"EVIDENCE:\n{evidence}\n\n"
        f"FINAL ANSWER:\n{t['final_answer']}"
    )

## 3. Ask Kev and decide

- `p(keep)` = calibrated probability that the answer is wrong or partial
- **keep** when `p(keep) >= KEEP_THRESHOLD`, or when the random-floor draw hits
- **severity high** when `p(keep) >= ALERT_THRESHOLD`

In [4]:
import math

@dataclass
class Decision:
    trace_id: str
    decision: str          # keep | drop
    p_keep: float          # calibrated
    p_keep_raw: float
    reason: str            # kev | random_floor
    severity: str          # high | normal | none
    model: str
    latency_ms: float

def calibrate(p, T):
    p = min(max(p, 1e-6), 1 - 1e-6)
    logit = math.log(p / (1 - p))
    return 1 / (1 + math.exp(-logit / T))

def ask_kev(trace):
    body = {"model": "kev-latest", "state": build_state(trace), "questions": QUESTIONS}
    t0 = time.perf_counter()
    r = requests.post(KEV_URL, json=body, timeout=TIMEOUT_S)
    r.raise_for_status()
    return r.json(), (time.perf_counter() - t0) * 1000

def decide(trace):
    resp, latency = ask_kev(trace)
    raw = resp["answers"][DECISION_QUESTION]["noul"]
    p_keep = calibrate(raw, TEMPERATURE)
    if p_keep >= KEEP_THRESHOLD:
        decision, reason = "keep", "kev"
    elif rng.random() < RANDOM_FLOOR:
        decision, reason = "keep", "random_floor"
    else:
        decision, reason = "drop", "kev"
    severity = "high" if p_keep >= ALERT_THRESHOLD else ("normal" if decision == "keep" else "none")
    return Decision(
        trace_id=trace.get("trace_id") or uuid.uuid4().hex,
        decision=decision, p_keep=round(p_keep, 4), p_keep_raw=round(raw, 4),
        reason=reason, severity=severity, model=resp.get("model", MODEL_NAME), latency_ms=round(latency, 1),
    )

## 4. Export kept traces to Phoenix

Each kept trace becomes one OpenTelemetry trace in Phoenix:
- a root **AGENT** span with the question as input and the final answer as output
- one **TOOL** span per tool call
- the decision on the root span as `eval.*` attributes, so you can filter and inspect it in Phoenix

The project name travels as the `openinference.project.name` resource attribute.

In [5]:
provider = TracerProvider(resource=Resource.create({"openinference.project.name": PHOENIX_PROJECT,
                                                    "service.name": "observer-ai-notebook"}))
provider.add_span_processor(SimpleSpanProcessor(OTLPSpanExporter(endpoint=PHOENIX_ENDPOINT)))
tracer = provider.get_tracer("observer-ai")

def export_to_phoenix(trace, d: Decision):
    with tracer.start_as_current_span("agent_run") as root:
        root.set_attribute("openinference.span.kind", "AGENT")
        root.set_attribute("input.value", trace["question"])
        root.set_attribute("output.value", trace["final_answer"])
        root.set_attribute("observer.source_trace_id", d.trace_id)
        for i, ev in enumerate(trace.get("evidence", [])):
            root.set_attribute(f"observer.evidence.{i}", ev[:2000])
        for key, value in asdict(d).items():
            root.set_attribute(f"eval.{key}", value)
        for c in trace.get("tool_calls", []):
            with tracer.start_as_current_span(c["name"]) as span:
                span.set_attribute("openinference.span.kind", "TOOL")
                span.set_attribute("tool.name", c["name"])
                span.set_attribute("input.value", json.dumps(c.get("args", "")))
                span.set_attribute("output.value", str(c.get("result_preview", c.get("status", "ok"))))
        phoenix_trace_id = format(root.get_span_context().trace_id, "032x")
    provider.force_flush()
    return phoenix_trace_id

## 5. Process one trace

The function the Go engine will mirror: take a trace, decide, and act only on **keep**.

In [6]:
def process_trace(trace: dict) -> Decision:
    d = decide(trace)
    if d.decision == "keep":
        phoenix_id = export_to_phoenix(trace, d)
        flag = "  ALERT (severity high)" if d.severity == "high" else ""
        print(f"KEEP  p(keep)={d.p_keep:.2f}  reason={d.reason}{flag}")
        print(f"      exported to Phoenix project '{PHOENIX_PROJECT}' as trace {phoenix_id}")
        print(json.dumps(asdict(d), indent=2))
    else:
        print(f"drop  p(keep)={d.p_keep:.2f}  ({d.latency_ms:.0f} ms)")
    return d

## 6. Try it

Two hand-made traces: one where the agent searched the wrong city, one that's correct.

In [7]:
wrong_trace = {
    "trace_id": "demo-wrong-001",
    "question": "Which river flows through the capital of the country where the Eiffel Tower is?",
    "tool_calls": [
        {"name": "search", "args": {"query": "Eiffel Tower location"}, "status": "ok"},
        {"name": "search", "args": {"query": "rivers in Lyon"}, "status": "ok"},
    ],
    "evidence": [
        "The Eiffel Tower is a wrought-iron lattice tower in Paris, France.",
        "Lyon sits at the confluence of the Rhône and Saône rivers.",
    ],
    "final_answer": "The Rhône",
}

correct_trace = {
    "trace_id": "demo-correct-001",
    "question": "Which river flows through the capital of the country where the Eiffel Tower is?",
    "tool_calls": [
        {"name": "search", "args": {"query": "Eiffel Tower location"}, "status": "ok"},
        {"name": "search", "args": {"query": "river through Paris"}, "status": "ok"},
    ],
    "evidence": [
        "The Eiffel Tower is a wrought-iron lattice tower in Paris, France.",
        "Paris, the capital of France, lies on the Seine.",
    ],
    "final_answer": "The Seine",
}

for t in (wrong_trace, correct_trace):
    print(f"--- {t['trace_id']}")
    process_trace(t)

--- demo-wrong-001
KEEP  p(keep)=0.65  reason=kev
      exported to Phoenix project 'observer-ai' as trace cc414b0f288c99936daba5af3b28d612
{
  "trace_id": "demo-wrong-001",
  "decision": "keep",
  "p_keep": 0.6495,
  "p_keep_raw": 0.6293,
  "reason": "kev",
  "severity": "normal",
  "model": "kev-latest",
  "latency_ms": 732.1
}
--- demo-correct-001
drop  p(keep)=0.07  (696 ms)


## 7. Optional: a handful of real held-out traces

Runs a few traces from the eval dataset and shows Kev's decisions next to the gold labels.

In [8]:
EVAL_PATH = Path("data/eval/labeled_traces.jsonl")

if EVAL_PATH.exists():
    rows = [json.loads(l) for l in EVAL_PATH.read_text().splitlines() if l.strip()]
    rows = [r for r in rows if r.get("split", "heldout") == "heldout"]
    sample = random.Random(7).sample(rows, min(10, len(rows)))
    hits = 0
    for r in sample:
        d = process_trace(r)
        truth = "keep" if r["label"] != "correct" else "drop"
        hits += d.decision == truth
        print(f"      gold: {r['label']} -> should {truth}\n")
    print(f"agreement with gold on this sample: {hits}/{len(sample)}")
else:
    print(f"{EVAL_PATH} not found, skipping")

KEEP  p(keep)=0.64  reason=kev
      exported to Phoenix project 'observer-ai' as trace 922b8aec4f935b8c9f9fefb22649b794
{
  "trace_id": "64f8faf5b71a4c5bbcde2e22df5682db",
  "decision": "keep",
  "p_keep": 0.6367,
  "p_keep_raw": 0.6181,
  "reason": "kev",
  "severity": "normal",
  "model": "kev-latest",
  "latency_ms": 124.6
}
      gold: wrong -> should keep

KEEP  p(keep)=0.54  reason=kev
      exported to Phoenix project 'observer-ai' as trace ee5d2d3a99ef0c469e162fc08b9a2cc2
{
  "trace_id": "a6fab4f682914e6cbea7157a7d0f1551",
  "decision": "keep",
  "p_keep": 0.544,
  "p_keep_raw": 0.5378,
  "reason": "kev",
  "severity": "normal",
  "model": "kev-latest",
  "latency_ms": 76.6
}
      gold: correct -> should drop

KEEP  p(keep)=0.77  reason=kev
      exported to Phoenix project 'observer-ai' as trace a34f2ddbc6c12e0666f1d4bb36b4aa9d
{
  "trace_id": "1f78eb3da0b1483ea9faa5016c3accd4",
  "decision": "keep",
  "p_keep": 0.7743,
  "p_keep_raw": 0.7423,
  "reason": "kev",
  "severity"

## 8. Check Phoenix

Open http://localhost:6006, pick the **observer-ai** project, and open a trace. The root span carries `eval.decision`, `eval.p_keep`, `eval.reason` and `eval.severity`.